# 08 Where is it? Maps — notebook

The Mayor's first question about any outbreak is *"where?"* In this notebook you will:

1. join dengue counts to the 24 barangay polygons of **Santa Luz City** (fictional; `santa_luz_barangays.geojson`) and compute **attack rates**;
2. map counts and rates side by side and see why they tell different stories;
3. choose **class breaks** and a **colour palette** on purpose;
4. see how unstable rates are when the numbers are small;
5. build an interactive map with plotly;
6. catch a join that silently loses two barangays;
7. zoom out to the real **18 regions of the Philippines** for context;
8. prepare a map that is safe to publish (small-cell suppression).

Everything about Santa Luz is invented; the regional map and its statistics are real and sourced.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

## Step 1 — Counts and rates per barangay

We need two things per barangay: the number of dengue cases (from the clean line list of [chapter 03](../03_clean_linelist.md)) and the population (from the barangay file). Then **attack rate per 10,000 = cases ÷ population × 10,000**.

> **Ask Tala:** *I have a GeoDataFrame `brgy` (24 rows; columns `barangay_id`, `barangay`, `band`, `population`, `geometry`) and a line list `ll` (one row per case; columns `barangay_id`, `disease`). Made-up example: `ll` rows `("SL01", "dengue")`, `("SL01", "dengue")`, `("SL02", "leptospirosis")`. Count dengue cases per `barangay_id`, join the counts to `brgy` **on `barangay_id`**, put 0 where a barangay has no case, and add `attack_rate` = cases / population × 10,000. The result must still have 24 rows.*

In [ ]:
import json

import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from scipy import stats

brgy = gpd.read_file("data/synthetic/ph/santa_luz_barangays.geojson")
pop = pd.read_csv("data/synthetic/ph/santa_luz_barangays.csv")[["barangay_id", "households"]]
ll = pd.read_csv("data/synthetic/ph/santa_luz_linelist.csv")

dengue_counts = ll[ll["disease"] == "dengue"].groupby("barangay_id").size().rename("cases")
brgy = brgy.merge(pop, on="barangay_id", how="left")
brgy = brgy.merge(dengue_counts, left_on="barangay_id", right_index=True, how="left")
brgy["cases"] = brgy["cases"].fillna(0).astype(int)        # a barangay with no case is 0, not missing
brgy["attack_rate"] = brgy["cases"] / brgy["population"] * 10_000

print(f"{len(brgy)} barangays, {brgy['cases'].sum()} dengue cases")
print(brgy.sort_values("attack_rate", ascending=False)
      [["barangay", "band", "population", "cases", "attack_rate"]].head(5).round(1).to_string(index=False))

**Check it.** (1) Still 24 rows after the joins? (2) Do the barangay counts add up to all 388 dengue cases in the line list? (3) One rate by hand: Talaba has 29 cases and 6,688 residents → 29 ÷ 6,688 × 10,000 = 43.4. (4) Is there a barangay with **zero** cases, and did it survive the join? `groupby` never creates a row for a barangay that has no cases — the `fillna(0)` above is what keeps it.

In [ ]:
assert len(brgy) == 24, "a join lost or duplicated barangays"
assert brgy["cases"].sum() == (ll["disease"] == "dengue").sum() == 388
talaba = brgy.set_index("barangay").loc["Talaba"]
assert abs(29 / 6688 * 10_000 - talaba["attack_rate"]) < 1e-9
print("Talaba by hand:", round(29 / 6688 * 10_000, 1))
print("Barangays with zero dengue cases:", brgy.loc[brgy["cases"] == 0, "barangay"].tolist())
print("Barangays in the groupby result:", len(dengue_counts), "<- one fewer than 24")

## Step 2 — Counts versus rates

Two maps of the same cases. On the left, **counts**: big barangays light up just because more people live there. On the right, **rates**: the chance that a resident became a case. Rates answer *"where is the risk?"*; counts answer *"where is the workload?"* Both are useful — just never mix them up.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, column, title in [(axes[0], "cases", "Dengue cases (count)"),
                          (axes[1], "attack_rate", "Attack rate per 10,000")]:
    brgy.plot(column=column, cmap="YlOrRd", edgecolor="#1c1b29", linewidth=0.6, legend=True,
              legend_kwds={"shrink": 0.6}, ax=ax)
    ax.set_title(title)
    ax.set_axis_off()
top_count = brgy.loc[brgy["cases"].idxmax()]
top_rate = brgy.loc[brgy["attack_rate"].idxmax()]
for ax, row in [(axes[0], top_count), (axes[1], top_rate)]:
    ax.annotate(row["barangay"], xy=(row.geometry.centroid.x, row.geometry.centroid.y),
                ha="center", fontsize=9, fontweight="bold",
                bbox={"boxstyle": "round,pad=0.2", "facecolor": "white", "alpha": 0.85})
fig.suptitle("Santa Luz City (fictional), dengue 20 Jul - 30 Sep 2026", fontsize=12)
plt.tight_layout()
plt.show()
print(f"Most cases: {top_count['barangay']} ({top_count['band']}, {top_count['cases']} cases, "
      f"{top_count['attack_rate']:.1f} per 10,000)")
print(f"Highest rate: {top_rate['barangay']} ({top_rate['band']}, {top_rate['cases']} cases, "
      f"{top_rate['attack_rate']:.1f} per 10,000)")

## Step 3 — Class breaks and colours

A choropleth groups values into a few **classes**, and the breaks change the message:

* **equal interval** — classes of equal width (here about 11 per 10,000 each); a few extreme barangays can leave most of the map in one colour;
* **quantile** — the same number of barangays in each class (24 ÷ 4 = 6); every colour is used, but neighbouring classes can be very close.

Colours: use a **sequential** palette (light = low, dark = high) such as `YlOrRd` or `viridis`; they read in order and survive colour-blindness and grey-scale printing. A **rainbow** palette (`jet`) has no natural order, and its bright yellow band in the middle draws the eye to *medium* values.

In [ ]:
rate = brgy["attack_rate"]
equal_edges = np.linspace(rate.min(), rate.max(), 5)          # 4 classes of equal width
quantile_edges = rate.quantile([0, 0.25, 0.5, 0.75, 1]).to_numpy()   # 4 classes of 6 barangays



def classes(values, edges, digits=1):
    # Cut values into classes labelled "low-high" (the lowest edge is included)
    labels = [f"{edges[i]:,.{digits}f}-{edges[i + 1]:,.{digits}f}" for i in range(len(edges) - 1)]
    return pd.cut(values, edges, include_lowest=True, labels=labels)


brgy["equal_class"] = classes(rate, equal_edges)
brgy["quantile_class"] = classes(rate, quantile_edges)
print("Equal interval:", brgy["equal_class"].value_counts(sort=False).to_dict())
print("Quantile:      ", brgy["quantile_class"].value_counts(sort=False).to_dict())

fig, axes = plt.subplots(1, 3, figsize=(15, 5.2))
panels = [("equal_class", "YlOrRd", "Equal interval, sequential"),
          ("quantile_class", "YlOrRd", "Quantile, sequential"),
          ("quantile_class", "jet", "Quantile, rainbow (avoid)")]
for ax, (column, cmap, title) in zip(axes, panels):
    brgy.plot(column=column, categorical=True, cmap=cmap, edgecolor="#1c1b29", linewidth=0.5,
              legend=True, legend_kwds={"loc": "upper center", "bbox_to_anchor": (0.5, 0.0), "ncol": 2,
                                        "fontsize": 8, "title": "per 10,000", "frameon": False}, ax=ax)
    ax.set_title(title)
    ax.set_axis_off()
plt.tight_layout()
plt.show()

## Step 4 — Small numbers jump around

The upland barangays have 3,000–5,000 residents and only a handful of cases, so one case more or less moves their rate a lot. An **exact Poisson 95% confidence interval** for the count shows how much: a wide interval means "don't rank this barangay on its colour alone".

In [ ]:
def poisson_ci(count, alpha=0.05):
    # Exact (Garwood) confidence interval for a Poisson count
    low = stats.chi2.ppf(alpha / 2, 2 * count) / 2 if count > 0 else 0.0
    high = stats.chi2.ppf(1 - alpha / 2, 2 * (count + 1)) / 2
    return low, high


ci = brgy["cases"].apply(lambda c: pd.Series(poisson_ci(c), index=["low", "high"]))
brgy["rate_low"] = ci["low"] / brgy["population"] * 10_000
brgy["rate_high"] = ci["high"] / brgy["population"] * 10_000
brgy["one_more_case"] = (brgy["cases"] + 1) / brgy["population"] * 10_000

cols = ["barangay", "band", "population", "cases", "attack_rate", "rate_low", "rate_high", "one_more_case"]
picked = brgy.set_index("barangay").loc[["Talaba", "Ilog", "Maligaya", "Bukid"]].reset_index()
print(picked[cols].round(1).to_string(index=False))

## Step 5 — An interactive map

For a screen (the dashboard of [chapter 09](../09_dashboard.md)), an interactive map lets the reader hover for exact numbers. Plotly needs the polygons as GeoJSON and the key that links each row to a polygon: `featureidkey="properties.barangay_id"`. We draw no background tiles: Santa Luz is fictional, and its coordinates deliberately sit in the open sea.

In [ ]:
geojson = json.loads(brgy[["barangay_id", "geometry"]].to_json())
fig = px.choropleth(
    brgy.drop(columns="geometry"),
    geojson=geojson,
    locations="barangay_id",
    featureidkey="properties.barangay_id",
    color="attack_rate",
    color_continuous_scale="YlOrRd",
    hover_name="barangay",
    hover_data={"band": True, "cases": True, "population": ":,", "attack_rate": ":.1f", "barangay_id": False},
    labels={"attack_rate": "per 10,000", "cases": "Cases", "population": "Population", "band": "Band"},
    title="Dengue attack rate per 10,000, Santa Luz City (fictional)",
)
fig.update_geos(fitbounds="locations", visible=False)
fig.update_layout(margin={"l": 0, "r": 0, "t": 40, "b": 0}, height=420)
fig.show()

## Tala got it wrong: the two barangays that vanished

Jun sent a quick table of dengue counts by barangay **name**, typed by hand. Maya asked Tala to join it to the map, and Tala joined **on the name**. The map looked fine — but Jun had typed `Santo Nino` (no ñ) and `Tabing Ilog` (no hyphen). With an inner join those two rows simply disappear; no error, no warning. Two riverside barangays with 41 cases between them would be blank on the Mayor's map.

The guard is one line: `assert len(merged) == 24`. The fix is to join on the **code** (`barangay_id`), never on a typed name.

In [ ]:
# Jun's hand-typed table (same counts, two names spelled differently)
jun = brgy[["barangay", "cases"]].copy()
jun["barangay"] = jun["barangay"].replace({"Santo Niño": "Santo Nino", "Tabing-Ilog": "Tabing Ilog"})

merged = brgy[["barangay_id", "barangay", "geometry"]].merge(jun, on="barangay", how="inner")   # Tala's join
print("Rows after Tala's join:", len(merged))
try:
    assert len(merged) == 24, f"join kept {len(merged)} of 24 barangays"
except AssertionError as err:
    print("Caught:", err)

check = brgy[["barangay"]].merge(jun, on="barangay", how="outer", indicator=True)
print(check[check["_merge"] != "both"].to_string(index=False))

## Step 6 — Zoom out: the 18 regions of the Philippines

For a briefing it helps to show where a province sits in the country. `ph_regions.topo.json` holds the real boundaries of the 18 regions as **TopoJSON** (a compact cousin of GeoJSON); geopandas reads it directly. We join the real regional statistics in `ph_regions_stats.csv` on the PSGC code `adm1_psgc` (a number in both files) and map **population density in 2024**. The density is derived (population 2024 ÷ land area), and both inputs carry a source URL in the CSV. NCR is about 22 times denser than the next region (CALABARZON), so with equal-width classes every other region would get the same colour; quantile classes show the differences.

This is a **context** map of the real country. Never draw fictional Santa Luz on it.

In [ ]:
regions = gpd.read_file("data/synthetic/ph/ph_regions.topo.json").set_crs("EPSG:4326")
ph_stats = pd.read_csv("data/synthetic/ph/ph_regions_stats.csv", dtype={"region_id": str, "psgc": str})
ph = regions[["adm1_psgc", "adm1_en", "geometry"]].merge(ph_stats, on="adm1_psgc", how="inner")
assert len(ph) == 18, f"join kept {len(ph)} of 18 regions"

density = ph["density_2024_per_km2"]
ph["density_class"] = classes(density, density.quantile([0, 0.25, 0.5, 0.75, 1]).to_numpy(), digits=0)
print(ph.sort_values("density_2024_per_km2", ascending=False)
      [["region_short", "population_2024", "land_area_km2", "density_2024_per_km2"]].head(4).to_string(index=False))
print("Source of population_2024:", ph["population_2024_source_url"].iloc[0])

fig, ax = plt.subplots(figsize=(6.5, 8.5))
ph.plot(column="density_class", categorical=True, cmap="YlGnBu", edgecolor="#1c1b29", linewidth=0.4,
        legend=True, legend_kwds={"loc": "upper right", "title": "People per km², 2024", "fontsize": 8}, ax=ax)
ax.set_title("Population density by region, Philippines, 2024")
ax.set_axis_off()
fig.text(0.02, 0.02, "Boundaries: NAMRIA, PSA via OCHA COD-AB (CC BY-IGO); TopoJSON: faeldon/philippines-json-maps (MIT).\n"
         "Density = 2024 population / land area, both from en.wikipedia.org/wiki/Regions_of_the_Philippines.",
         fontsize=6.5)
plt.tight_layout()
plt.show()

## Step 7 — Make it safe to publish

A **dot map** of households with cases is a list of addresses drawn as dots: anyone who knows the street can point at a house. Inside the CESU it may guide the 4S teams; on a public page it identifies neighbours. Public maps show **aggregated polygons** (barangays), and even then a barangay with 1–4 cases is a small cell that can point to a person. A common rule, and the one this book uses: **suppress counts of 1–4** (show "<5") and suppress their rates too, because rate × population gives the count back.

In [ ]:
public = brgy[["barangay_id", "barangay", "population", "cases", "attack_rate", "geometry"]].copy()
small = public["cases"].between(1, 4)
public["cases_public"] = public["cases"].astype(str).where(~small, "<5")
public["rate_public"] = public["attack_rate"].round(1).where(~small)        # NaN = not shown
print(f"Suppressed: {small.sum()} of 24 barangays:", public.loc[small, "barangay"].tolist())

fig, ax = plt.subplots(figsize=(7, 4.8))
public.plot(column="rate_public", cmap="YlOrRd", edgecolor="#1c1b29", linewidth=0.5, legend=True,
            legend_kwds={"shrink": 0.6, "label": "per 10,000"}, ax=ax,
            missing_kwds={"color": "#eeeeee", "hatch": "///", "edgecolor": "#6e7882", "label": "1-4 cases: not shown"})
ax.set_title("Public version: dengue per 10,000 (cells of 1-4 cases suppressed)")
ax.set_axis_off()
plt.tight_layout()
plt.show()

## Your turn

1. **Leptospirosis.** Repeat Step 1 for `disease == "leptospirosis"` and map the rate. Which band lights up now? *(Hint: change one string, and remember the `fillna(0)`.)*
2. **Five classes.** Redo Step 3 with 5 quantile classes. Does Talaba still sit alone in the top class? *(Hint: `rate.quantile(np.linspace(0, 1, 6))`.)*
3. **A different regional variable.** Map `poverty_incidence_2023_pct` instead of density, and add its source URL from the `poverty_incidence_2023_pct_source_url` column to the credit line.

In [ ]:
# 1. Leptospirosis (starter)
lepto = ll[ll["disease"] == "leptospirosis"].groupby("barangay_id").size().rename("lepto_cases")
brgy_l = brgy[["barangay_id", "barangay", "band", "population", "geometry"]].merge(
    lepto, left_on="barangay_id", right_index=True, how="left")
brgy_l["lepto_cases"] = brgy_l["lepto_cases"].fillna(0).astype(int)
brgy_l["lepto_rate"] = brgy_l["lepto_cases"] / brgy_l["population"] * 10_000
assert len(brgy_l) == 24
by_band = brgy_l.groupby("band")[["lepto_cases", "population"]].sum()   # add up first: a mean of rates is not the band's rate
print((by_band["lepto_cases"] / by_band["population"] * 10_000).round(1))

## What you did

* Joined counts to polygons **on a code**, kept zero-case barangays, and asserted 24 rows.
* Mapped rates, not counts, when the question is risk; chose class breaks and a sequential palette on purpose.
* Read wide confidence intervals as "do not rank on colour alone".
* Drew the real 18-region context map with its credit line.
* Prepared a public map with small cells suppressed.

Next: [chapter 09](../09_dashboard.md) puts the curve, the threshold and this map on one screen.